# Neuronales Netz - Aktivierungsfunktion, Lernrate, Batch-Size, Epochenanzahl

**systematische Einschränkung der Hyperparameterbreiche**

**ki-308** California House Prising

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LinearRegression

from utils.data import load_and_clean_data, get_train_test_split, Standard_Scaler, Min_Max_Scaler
from utils.evaluation import evaluate_predictions, add_result
from utils.plotting import plot_predicted_vs_actual, plot_residuals, save_fig

plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print(f"TensorFlow Version: {tf.__version__}")
print(f"Numpy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print(f"python Version: {sys.version}")

In [ ]:
def build_model(hidden_layers, activation='relu', learning_rate=0.001):
    """Erstelle ein Sequential-Modell mit gegebener Architektur."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    
    model.add(layers.Dense(1))  # Regression Output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


def train_and_evaluate(model, name, x_train, y_train, x_val, y_val, x_test, y_test,
                       epochs=100, batch_size=32, verbose=0):

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )

    y_train_pred = model.predict(x_train, verbose=0).flatten()
    y_test_pred = model.predict(x_test, verbose=0).flatten()

    result = evaluate_predictions(y_train, y_train_pred, y_test, y_test_pred, name)
    add_result(result)

    return history, result

## Daten laden und transformieren

Da wir sehr verschiedene Intervalle haben sollte eine Skalierung sehr sinnvoll sein, um ansonsten z.B. Sigmoid nicht gut funktionieren kann.

In [ ]:
df = load_and_clean_data()
X_train_standard, X_test_standard, y_train_standard, y_test_standard, scaler, feature_names = get_train_test_split(df, scaler='standard')

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standar.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")

np.isnan(X_train_min0_max1).sum()
np.isnan(X_val_min0_max1).sum()
np.isnan(X_test_min0_max1).sum()

Beginn mit ReLu: (https://www.datacamp.com/de/tutorial/introduction-to-activation-functions-in-neural-networks?dc_referrer=https%3A%2F%2Fwww.google.com%2F)

- rechnerisch kostengünstig auch bei Anwendung mehrerer Schichten
- gängiste Aktivierungsfunktion

Fixparameter:
- Loss: MSE
- Hidden Layers: 2 mit 64 und 32 units (Datensatz zu klein für tieferes Netz und zu geringe Features für breiteres Netz; abnehmende Größen beugen Overfitting vor [Copilot Promt: welche fixparameter wie layer-, unitanzahl etc. sollte man hier wählen?])
- Metric: mae
- learning_rate: 0.001
- Epochen: 100 (von Florian übernommen)
- Batch-Size: 32 (von Florian übernommen)

In [ ]:
model = keras.Sequential()
model.add(layers.Input(shape=(X_train_min0_max1.shape[1],)))
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(32, activation="relu"))
model.add(layers.Dense(1))  # Regression Output
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)
train_and_evaluate(model, "NN_min0_max1_relu", X_train_min0_max1, y_train_min0_max1, X_val_min0_max1, y_val_min0_max1, X_test_min0_max1, y_test_min0_max1, epochs=100, batch_size=32, verbose=0)
